# Elastic Stack on Docker — Complete Deployment Guide

**Stack Version:** `9.4.1`  
**Environment:** Ubuntu VM · Single-Node Docker Cluster  
**Components:** Elasticsearch · Kibana · Fleet Server · Elastic Agent  
**Author:** Santanu  
**Purpose:** Reproducible lab/testing deployment reference

---

## What This Notebook Covers

| Section | Description |
|---|---|
| **Part 1** | Prerequisites & environment setup |
| **Part 2** | Install Elasticsearch in Docker (official method) |
| **Part 3** | Install Kibana in Docker |
| **Part 4** | Activate Trial License |
| **Part 5** | Configure Kibana for Fleet |
| **Part 6** | Deploy Fleet Server |
| **Part 7** | Install Elastic Agent on a remote host |
| **Part 8** | Add System & Nginx Integrations |
| **Part 9** | Daily Start / Stop Operations |
| **Part 10** | Troubleshooting Reference |

---

> ⚠️ **How to use this notebook:**  
> Commands prefixed with `# [RUN ON VM]` run on the Ubuntu VM hosting Docker.  
> Commands prefixed with `# [RUN ON AGENT HOST]` run on the remote Ubuntu host where you want to install the Elastic Agent.  
> All `curl` verification commands run on the VM unless stated otherwise.

---
# Part 1 — Prerequisites

Before starting, ensure the following are in place on your Ubuntu VM.

## 1.1 — Install Docker Engine (if not already installed)

Follow the official Docker install for Ubuntu. The commands below are for reference — run them manually if Docker is not yet installed.

In [ ]:
# [RUN ON VM] — Install Docker Engine on Ubuntu
# Skip if Docker is already installed

sudo apt-get update
sudo apt-get install -y ca-certificates curl gnupg
sudo install -m 0755 -d /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /etc/apt/keyrings/docker.gpg
sudo chmod a+r /etc/apt/keyrings/docker.gpg

echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] \
  https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

sudo apt-get update
sudo apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin

In [ ]:
# [RUN ON VM] — Verify Docker installation
docker --version
sudo systemctl status docker --no-pager | head -5

## 1.2 — Set Environment Variables

Set the Elastic Stack version. Update this value to change the version for the entire deployment.

In [ ]:
# [RUN ON VM] — Set stack version (update this to change version globally)
export ELASTIC_VERSION="9.4.1"
echo "Elastic Stack version set to: $ELASTIC_VERSION"

## 1.3 — Set VM Max Map Count (Required for Elasticsearch)

In [ ]:
# [RUN ON VM] — Increase vm.max_map_count for Elasticsearch
sudo sysctl -w vm.max_map_count=262144

# Make it persistent across reboots
echo 'vm.max_map_count=262144' | sudo tee -a /etc/sysctl.conf

# Verify
sysctl vm.max_map_count

---
# Part 2 — Install Elasticsearch in Docker

Following the **official Elastic single-node Docker method**.

## 2.1 — Create Docker Network

In [ ]:
# [RUN ON VM] — Create dedicated Docker network for Elastic Stack
sudo docker network create elastic

# Verify network created
sudo docker network ls | grep elastic

## 2.2 — Pull Elasticsearch Image

In [ ]:
# [RUN ON VM] — Pull Elasticsearch Docker image
sudo docker pull docker.elastic.co/elasticsearch/elasticsearch:9.4.1

## 2.3 — Start Elasticsearch Container

> ⚠️ **IMPORTANT:** When this container starts for the first time, it prints the `elastic` user password and a Kibana enrollment token to the terminal output. **Save both values immediately.**

In [ ]:
# [RUN ON VM] — Start Elasticsearch container
# Run this in a terminal (not as a background process on first run)
# so you can capture the password and enrollment token from the output

sudo docker run --name es01 \
  --net elastic \
  -p 9200:9200 \
  -it \
  -m 6GB \
  -e "xpack.ml.use_auto_machine_memory_percent=true" \
  docker.elastic.co/elasticsearch/elasticsearch:9.4.1

# NOTE: After capturing the password and enrollment token,
# press Ctrl+C and then start it detached:
# sudo docker start es01

## 2.4 — Save / Reset the elastic Password

If you missed the password from the first run output, reset it:

In [ ]:
# [RUN ON VM] — Reset the elastic user password (run if you missed the initial output)
sudo docker exec -it es01 \
  /usr/share/elasticsearch/bin/elasticsearch-reset-password -u elastic

# Save the output password — you will need it throughout this guide
# Example: ES_PASSWORD="your_password_here"
# Replace with your actual password below:
export ES_PASSWORD="REPLACE_WITH_YOUR_PASSWORD"
echo "Password saved to ES_PASSWORD variable"

## 2.5 — Verify Elasticsearch is Running

In [ ]:
# [RUN ON VM] — Verify Elasticsearch cluster health
# Replace ES_PASSWORD with your actual password if not set as environment variable

curl -k -u elastic:${ES_PASSWORD} \
  https://localhost:9200/_cluster/health?pretty

# Expected: "status" : "green" or "yellow"
# green  = all primary and replica shards assigned
# yellow = all primary shards assigned, replica shards not yet assigned (normal for single-node)

In [ ]:
# [RUN ON VM] — Verify Elasticsearch version and node info
curl -k -u elastic:${ES_PASSWORD} https://localhost:9200?pretty

---
# Part 3 — Install Kibana in Docker

## 3.1 — Pull Kibana Image

In [ ]:
# [RUN ON VM] — Pull Kibana Docker image
sudo docker pull docker.elastic.co/kibana/kibana:9.4.1

## 3.2 — Start Kibana Container

> ℹ️ The three encryption keys are **required** for Fleet to work. Use any random 32+ character string. Keep them consistent across restarts.

In [ ]:
# [RUN ON VM] — Start Kibana container with required encryption keys
# The encryption keys must be at least 32 characters
# You can generate your own or use the ones below for testing

sudo docker run --name kib01 \
  --net elastic \
  -p 5601:5601 \
  -e XPACK_ENCRYPTEDSAVEDOBJECTS_ENCRYPTIONKEY="a7a6311933d3503b89bc2dbc36572c33a6c10925682e591bffcab6911c06786d" \
  -e XPACK_REPORTING_ENCRYPTIONKEY="a7a6311933d3503b89bc2dbc36572c33a6c10925682e591bffcab6911c06786d" \
  -e XPACK_SECURITY_ENCRYPTIONKEY="a7a6311933d3503b89bc2dbc36572c33a6c10925682e591bffcab6911c06786d" \
  -d \
  docker.elastic.co/kibana/kibana:9.4.1

echo "Kibana container started. Wait 60 seconds before verifying."

## 3.3 — Verify Kibana is Running

In [ ]:
# [RUN ON VM] — Check Kibana status (wait 60 seconds after starting before running)
curl -s http://localhost:5601/api/status | python3 -m json.tool | grep -A3 '"overall"'

# Expected: "level": "available"
# If you see "unavailable" wait another 30 seconds and retry

In [ ]:
# [RUN ON VM] — Verify both containers are running
sudo docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

# Expected output:
# NAMES    STATUS         PORTS
# kib01    Up X minutes   0.0.0.0:5601->5601/tcp
# es01     Up X minutes   0.0.0.0:9200->9200/tcp

## 3.4 — Log into Kibana

Open your browser and navigate to: **http://localhost:5601**

| Field | Value |
|---|---|
| **Username** | `elastic` |
| **Password** | *(the password saved in Step 2.4)* |

---
# Part 4 — Activate Trial License

The default installation starts with a **Basic license** which does not include Fleet or advanced Observability features. Activate the 30-day trial to unlock all features.

In [ ]:
# [RUN ON VM] — Check current license type
curl -k -u elastic:${ES_PASSWORD} \
  https://localhost:9200/_license?pretty

# If "type": "basic" → proceed to activate trial below
# If "type": "trial" → already activated, skip next cell

In [ ]:
# [RUN ON VM] — Activate 30-day trial license
curl -k -u elastic:${ES_PASSWORD} \
  -X POST \
  "https://localhost:9200/_license/start_trial?acknowledge=true"

# Expected response:
# { "acknowledged": true, "trial_was_started": true, "type": "trial" }

In [ ]:
# [RUN ON VM] — Verify trial is active and check expiry date
curl -k -u elastic:${ES_PASSWORD} \
  https://localhost:9200/_license?pretty

# Expected:
# "type" : "trial"
# "status" : "active"
# "expiry_date" : "YYYY-MM-DDT..."  ← 30 days from activation

---
# Part 5 — Configure Kibana for Fleet

These steps are performed in the **Kibana UI**. Follow the instructions in each cell.

## 5.1 — Initialize Fleet in Kibana

In Kibana UI:
1. Go to **Management → Fleet**
2. Click **Get started** if prompted
3. Fleet will initialize — this may take 30–60 seconds

> ⚠️ If you see **"Unable to initialize Fleet — Agent binary source needs encrypted saved object api key"**, it means the encryption keys were not set. Stop the kib01 container and re-run Step 3.2 with the encryption key environment variables.

## 5.2 — Configure Fleet Settings

In Kibana UI go to **Management → Fleet → Settings**:

| Setting | Value |
|---|---|
| **Fleet Server hosts** | `https://fleet-server:8220` |
| **Elasticsearch output host** | *(leave as default — `https://172.18.0.x:9200`)* |

Click **Save and apply settings**.

> ℹ️ The default Elasticsearch output uses the internal Docker IP (e.g. `172.18.0.2`). This is correct for agents running **inside** the Docker network. For agents on **external hosts**, we create a separate output in Part 8.

---
# Part 6 — Deploy Fleet Server

Fleet Server acts as the communication hub between Kibana and all Elastic Agents.

## 6.1 — Generate Fleet Server Service Token

In [ ]:
# [RUN ON VM] — Generate a service token for Fleet Server
# If this returns a 409 conflict (token already exists), change fleet-token-1 to fleet-token-2, etc.

curl -k -u elastic:${ES_PASSWORD} \
  -X POST \
  "https://localhost:9200/_security/service/elastic/fleet-server/credential/token/fleet-token-1"

# Save the "value" field from the response — this is your SERVICE_TOKEN
# Example output:
# { "created": true, "token": { "name": "...", "value": "AAEAAWVsY..." } }

# Set it as a variable (replace with your actual token value):
export SERVICE_TOKEN="REPLACE_WITH_TOKEN_VALUE"
echo "Service token saved"

## 6.2 — Pull Elastic Agent Image

In [ ]:
# [RUN ON VM] — Pull the Elastic Agent image (used for Fleet Server)
# Note: In Elastic 9.x the image path is elastic-agent/elastic-agent (not beats/elastic-agent)

sudo docker pull docker.elastic.co/elastic-agent/elastic-agent:9.4.1

## 6.3 — Start Fleet Server Container

> ℹ️ **Key flags explained:**
> - `FLEET_SERVER_HOST=0.0.0.0` — binds Fleet Server to all interfaces (required for external agents to connect)
> - `FLEET_SERVER_PORT=8220` — explicit port binding (required in 9.x)
> - `FLEET_SERVER_ELASTICSEARCH_INSECURE=true` — accepts self-signed cert from Elasticsearch
> - `FLEET_INSECURE=true` — allows insecure connections from agents

In [ ]:
# [RUN ON VM] — Start Fleet Server container
# Replace SERVICE_TOKEN with your actual token value if not set as environment variable

sudo docker run --name fleet-server \
  --net elastic \
  -p 8220:8220 \
  -e FLEET_SERVER_ENABLE=true \
  -e FLEET_SERVER_ELASTICSEARCH_HOST=https://es01:9200 \
  -e FLEET_SERVER_ELASTICSEARCH_INSECURE=true \
  -e FLEET_SERVER_SERVICE_TOKEN=${SERVICE_TOKEN} \
  -e FLEET_SERVER_HOST=0.0.0.0 \
  -e FLEET_SERVER_PORT=8220 \
  -e FLEET_INSECURE=true \
  -d \
  docker.elastic.co/elastic-agent/elastic-agent:9.4.1

echo "Fleet Server container started. Wait 60 seconds before verifying."

## 6.4 — Verify Fleet Server

In [ ]:
# [RUN ON VM] — Verify Fleet Server HTTP status (wait 60 seconds after starting)
curl http://localhost:8220/api/status

# Expected: {"name":"fleet-server","status":"HEALTHY"}

In [ ]:
# [RUN ON VM] — Check Fleet Server logs for HEALTHY confirmation
sudo docker logs fleet-server 2>&1 | grep -i 'healthy\|error\|fatal' | tail -20

In [ ]:
# [RUN ON VM] — Verify all three containers are running
sudo docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

# Expected:
# NAMES          STATUS         PORTS
# fleet-server   Up X minutes   0.0.0.0:8220->8220/tcp
# kib01          Up X minutes   0.0.0.0:5601->5601/tcp
# es01           Up X minutes   0.0.0.0:9200->9200/tcp

## 6.5 — Verify Fleet Server in Kibana UI

In Kibana go to **Management → Fleet → Agents**

You should see **Fleet Server** listed as an agent with a green **Healthy** badge.

> ⚠️ If Fleet shows **"Unable to initialize Fleet"** after a restart, hard-refresh the browser (`Ctrl+Shift+R`) and navigate to Fleet again. If it persists, restart Kibana: `sudo docker restart kib01`

---
# Part 7 — Install Elastic Agent on a Remote Ubuntu Host

This section covers installing the Elastic Agent on a **separate Ubuntu machine** (e.g. a host running NGINX) and enrolling it into Fleet.

> **VM IP Address:** Replace `<VM_IP>` throughout this section with the actual IP of your Docker VM (e.g. `172.20.58.140`)

## 7.1 — Prerequisites on the Remote Host

Before installing the agent, ensure the remote host can reach the VM.

In [ ]:
# [RUN ON AGENT HOST] — Test connectivity to Fleet Server on the VM
# Replace <VM_IP> with your actual VM IP address

curl http://<VM_IP>:8220/api/status

# Expected: {"name":"fleet-server","status":"HEALTHY"}
# If connection refused: check VM firewall (sudo ufw allow 8220/tcp on the VM)

## 7.2 — Add DNS Entry on the Remote Host

Fleet Server uses the hostname `fleet-server` internally. Add a local hosts entry so the agent can resolve it.

> ℹ️ In production this would be a proper DNS record. For lab use, `/etc/hosts` is the equivalent.

In [ ]:
# [RUN ON AGENT HOST] — Add fleet-server hostname to /etc/hosts
# Replace <VM_IP> with your actual VM IP address

echo "<VM_IP>    fleet-server" | sudo tee -a /etc/hosts

# Verify the entry was added
grep fleet-server /etc/hosts

# Test hostname resolution
curl http://fleet-server:8220/api/status
# Expected: {"name":"fleet-server","status":"HEALTHY"}

## 7.3 — Get Enrollment Token from Kibana

In Kibana UI go to:

**Management → Fleet → Enrollment Tokens → Create enrollment token**

| Field | Value |
|---|---|
| **Name** | `ubuntu-agent-token` (or any descriptive name) |
| **Policy** | `Default policy` |

Copy the token value — you will use it in the install command below.

## 7.4 — Download and Install Elastic Agent

In [ ]:
# [RUN ON AGENT HOST] — Download Elastic Agent
curl -L -O \
  https://artifacts.elastic.co/downloads/beats/elastic-agent/elastic-agent-9.4.1-linux-x86_64.tar.gz

# Extract
tar xzvf elastic-agent-9.4.1-linux-x86_64.tar.gz

# Enter the directory
cd elastic-agent-9.4.1-linux-x86_64

echo "Elastic Agent downloaded and extracted"

In [ ]:
# [RUN ON AGENT HOST] — Install and enroll Elastic Agent
# Replace <ENROLLMENT_TOKEN> with the token copied from Kibana in Step 7.3

sudo ./elastic-agent install \
  --url=http://fleet-server:8220 \
  --enrollment-token=<ENROLLMENT_TOKEN> \
  --insecure

# Expected final output:
# Successfully enrolled the Elastic Agent.
# Elastic Agent has been successfully installed.

## 7.5 — Reinstall Notes

If the agent was previously installed and you need to reinstall:

In [ ]:
# [RUN ON AGENT HOST] — Uninstall existing agent before reinstalling
# Only run this if the agent was previously installed

sudo elastic-agent uninstall
# Type Y when prompted

# Verify removed
sudo ls /opt/Elastic/Agent 2>/dev/null || echo "Agent successfully removed"

## 7.6 — Verify Agent Enrollment

In [ ]:
# [RUN ON AGENT HOST] — Check Elastic Agent status
sudo elastic-agent status

# Expected output:
# ┌─ fleet
# │  └─ status: (HEALTHY) Connected
# └─ elastic-agent
#    └─ status: (HEALTHY) Running

---
# Part 8 — Add Integrations and Configure External Output

For agents on **external hosts** (outside Docker), you need a separate Elasticsearch output that uses the VM's real IP instead of the internal Docker IP.

## 8.1 — Create External Elasticsearch Output in Kibana

In Kibana UI go to **Management → Fleet → Settings → Outputs → Add output**:

| Field | Value |
|---|---|
| **Name** | `external-output` |
| **Type** | `Elasticsearch` |
| **Hosts** | `https://<VM_IP>:9200` |
| **Advanced YAML** | `ssl.verification_mode: none` |

Click **Save and apply settings**.

> ℹ️ The `ssl.verification_mode: none` is required because Elasticsearch uses a self-signed certificate. In production you would provide the actual CA certificate instead.

## 8.2 — Assign External Output to Agent Policy

In Kibana UI go to **Management → Fleet → Agent policies → Default policy → Settings**:

| Field | Value |
|---|---|
| **Output for integrations** | `external-output` |
| **Output for agent monitoring** | `external-output` |

Click **Save**.

## 8.3 — Add System and Nginx Integrations

In Kibana UI go to **Management → Fleet → Agent policies → Default policy → Add integration**

### System Integration
- Search **System** → Click **Add System**
- Keep default settings (collects CPU, memory, disk metrics + system/auth logs)
- Click **Save and continue**

### Nginx Integration
- Click **Add integration** again
- Search **Nginx** → Click **Add Nginx**
- Default log paths: `/var/log/nginx/access.log` and `/var/log/nginx/error.log`
- Click **Save and continue**

> ℹ️ **Why not use Auto-detect?** The auto-detect method picks up ALL integrations it finds on the host (including Chrome, etc.). Manual integration selection gives you precise control over what data is collected.

In [ ]:
# [RUN ON AGENT HOST] — Restart agent after adding integrations
sudo systemctl restart elastic-agent

# Wait 30 seconds then verify
sleep 30
sudo elastic-agent status

# All components should show HEALTHY:
# system/metrics-default    → HEALTHY
# nginx/metrics-default     → HEALTHY
# filestream-monitoring     → HEALTHY

## 8.4 — Verify Data in Kibana

After 2–3 minutes, check the following in Kibana:

| View | Path |
|---|---|
| **Live log stream** | Observability → Logs → Stream |
| **Host metrics** | Observability → Infrastructure → Hosts |
| **Nginx dashboard** | Dashboards → search "Nginx" |
| **System dashboard** | Dashboards → search "System" |

---
# Part 9 — Daily Start / Stop Operations

Follow these procedures every time you stop or start the VM.

## 9.1 — Stopping the Stack (Before VM Shutdown)

> ⚠️ Always stop in this order: **Fleet Server → Kibana → Elasticsearch**

In [ ]:
# [RUN ON VM] — Stop all containers in correct order
sudo docker stop fleet-server
sudo docker stop kib01
sudo docker stop es01

# Verify all stopped
sudo docker ps
# Expected: no containers listed (empty output)

In [ ]:
# [RUN ON VM] — Alternative: Stop all in one command
sudo docker stop fleet-server kib01 es01

## 9.2 — Starting the Stack (Next Day)

> ⚠️ Always start in this order: **Elasticsearch → Kibana → Fleet Server**

In [ ]:
# [RUN ON VM] — Step 1: Start Elasticsearch
sudo docker start es01

# Wait 30 seconds then verify
sleep 30
curl -k -u elastic:${ES_PASSWORD} \
  https://localhost:9200/_cluster/health?pretty | grep status

# Expected: "status" : "green" or "yellow"
# Do NOT proceed if status is "red"

In [ ]:
# [RUN ON VM] — Step 2: Start Kibana
sudo docker start kib01

# Wait 60 seconds then verify
sleep 60
curl -s http://localhost:5601/api/status | python3 -m json.tool | grep -A3 '"overall"'

# Expected: "level": "available"

In [ ]:
# [RUN ON VM] — Step 3: Start Fleet Server
sudo docker start fleet-server

# Wait 30 seconds then verify
sleep 30
curl http://localhost:8220/api/status

# Expected: {"name":"fleet-server","status":"HEALTHY"}

In [ ]:
# [RUN ON VM] — Verify all three containers are running
sudo docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

# Expected: All three showing "Up"
# fleet-server   Up X seconds   0.0.0.0:8220->8220/tcp
# kib01          Up X minutes   0.0.0.0:5601->5601/tcp
# es01           Up X minutes   0.0.0.0:9200->9200/tcp

In [ ]:
# [RUN ON VM] — Alternative: Start all in one command
# Note: Docker does not wait for each container to be ready before starting the next
# If Kibana or Fleet Server fail, restart them individually after ES is confirmed healthy

sudo docker start es01 kib01 fleet-server

---
# Part 10 — Troubleshooting Reference

Common issues and their fixes.

In [ ]:
# TROUBLESHOOTING — View Elasticsearch logs
sudo docker logs es01 --tail 50 2>&1 | grep -i 'error\|warn\|started\|failed'

In [ ]:
# TROUBLESHOOTING — View Kibana logs
sudo docker logs kib01 --tail 50 2>&1 | grep -i 'error\|warn\|started\|failed'

In [ ]:
# TROUBLESHOOTING — View Fleet Server logs
sudo docker logs fleet-server --tail 50 2>&1 | grep -i 'error\|healthy\|failed\|fatal'

In [ ]:
# TROUBLESHOOTING — Kibana shows 'server is not ready yet' after restart
sudo docker restart kib01
sleep 60
curl -s http://localhost:5601/api/status | python3 -m json.tool | grep level

In [ ]:
# TROUBLESHOOTING — Fleet Server not HEALTHY after restart
sudo docker restart fleet-server
sleep 45
curl http://localhost:8220/api/status

In [ ]:
# TROUBLESHOOTING — Check license status and expiry
curl -k -u elastic:${ES_PASSWORD} \
  https://localhost:9200/_license?pretty | grep -E 'type|status|expiry_date'

# If type is "basic" — Fleet features won't work. Run activate trial cell in Part 4.

In [ ]:
# TROUBLESHOOTING — [RUN ON AGENT HOST] Agent degraded after VM restart
# Check agent status
sudo elastic-agent status

# If degraded, restart the agent service
sudo systemctl restart elastic-agent
sleep 30
sudo elastic-agent status

In [ ]:
# TROUBLESHOOTING — [RUN ON VM] Open firewall ports if agent cannot reach VM
sudo ufw status

# If active, open required ports:
sudo ufw allow 9200/tcp   # Elasticsearch
sudo ufw allow 5601/tcp   # Kibana
sudo ufw allow 8220/tcp   # Fleet Server

sudo ufw status

In [ ]:
# TROUBLESHOOTING — Full stack health check (run all at once)
echo "=== Docker Containers ==="
sudo docker ps --format "table {{.Names}}\t{{.Status}}"

echo ""
echo "=== Elasticsearch ==="
curl -sk -u elastic:${ES_PASSWORD} https://localhost:9200/_cluster/health | python3 -m json.tool | grep -E 'status|number_of_nodes'

echo ""
echo "=== Kibana ==="
curl -s http://localhost:5601/api/status | python3 -m json.tool | grep level | head -3

echo ""
echo "=== Fleet Server ==="
curl -s http://localhost:8220/api/status

echo ""
echo "=== License ==="
curl -sk -u elastic:${ES_PASSWORD} https://localhost:9200/_license | python3 -m json.tool | grep -E 'type|status|expiry_date'

---
# Quick Reference — All Key Commands

| Operation | Command |
|---|---|
| Stop all (ordered) | `sudo docker stop fleet-server kib01 es01` |
| Start all (ordered) | `sudo docker start es01 kib01 fleet-server` |
| All container status | `sudo docker ps` |
| ES cluster health | `curl -k -u elastic:<PW> https://localhost:9200/_cluster/health?pretty` |
| Fleet Server status | `curl http://localhost:8220/api/status` |
| Kibana status | `curl -s http://localhost:5601/api/status` |
| License check | `curl -k -u elastic:<PW> https://localhost:9200/_license?pretty` |
| ES logs | `sudo docker logs es01 --tail 50` |
| Kibana logs | `sudo docker logs kib01 --tail 50` |
| Fleet logs | `sudo docker logs fleet-server --tail 50` |
| Restart container | `sudo docker restart <name>` |
| Agent status (remote) | `sudo elastic-agent status` |
| Restart agent (remote) | `sudo systemctl restart elastic-agent` |
| Uninstall agent (remote) | `sudo elastic-agent uninstall` |

---

## Important Notes

- **Trial License** expires after 30 days. After expiry, Fleet features stop working. To renew, recreate the `es01` container with a fresh data volume.
- **Enrollment tokens** expire. Always generate a fresh token from **Fleet → Enrollment Tokens** when re-enrolling an agent.
- **Service token** (`fleet-token-x`) persists in Elasticsearch. If you need a new one, increment the name (e.g. `fleet-token-3`).
- **External agents** require the `external-output` pointing to `https://<VM_IP>:9200` with `ssl.verification_mode: none` in the output YAML.
- **Fleet Server** in this lab runs on plain **HTTP** (port 8220). The `--insecure` flag is required when enrolling agents.